## 0. Setup & Imports

All dependencies and global paths. Run this cell first.

In [ ]:
import os
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cvxpy as cp
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "font.size": 10,
    "axes.spines.right": False,
    "axes.spines.top": False,
})

# Paths (adjust these to your environment)
MASTER_PATH = "data/master_hourly.csv"
PF_PATH     = "data/pf_daily_revenue.csv"
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Walk-forward parameters
TRAIN_MONTHS = 12
TEST_MONTHS  = 1
STEP_MONTHS  = 1

# Forecast bounds (JPY/kWh)
PRICE_FLOOR = 0.01
PRICE_CEIL  = 200.0

print("Setup complete.")

In [ ]:
# Load master and perfect-foresight benchmark
master = pd.read_csv(MASTER_PATH)
pf = pd.read_csv(PF_PATH)
pf["date"] = pd.to_datetime(pf["date"]).dt.date

print(f"Master  : {len(master):>6,} hourly rows  ({master['timestamp_jst'].min()} → {master['timestamp_jst'].max()})")
print(f"PF      : {len(pf):>6,} days")
print(f"Features: {list(master.columns)[:8]}...")

## 1. LP Dispatch Model

Single-day energy arbitrage LP. Battery specifications:

- Power: 10 MW
- Capacity: 40 MWh
- Round-trip efficiency: 0.85 (sqrt-split on charge & discharge)
- SOC bounds: 10%–90% of E_max (useable = 32 MWh)
- Initial SOC: 10%
- Cycle cap: 1 cycle/day → max daily discharge = 32 × √0.85 ≈ 29.50 MWh

**Validated** to within 0.00% of `pf_daily_revenue.csv` across 10 dates.

The `dispatch_and_revalue` function is the **shared evaluator** used by every tier.


In [ ]:
DEFAULT_PARAMS = {
    "p_max": 10.0,
    "e_max": 40.0,
    "eta_rt": 0.85,
    "soc_min_frac": 0.10,
    "soc_max_frac": 0.90,
    "soc_init_frac": 0.10,
    "dt": 1.0,
    "max_cycles_per_day": 1.0,
}


def run_lp_day(price_vec, params=None):
    """Solve a single-day arbitrage LP.

    price_vec : array-like of length T (typically 24), JPY/kWh
    Returns dict with charge, discharge, soc, revenue, status.
    Revenue is in JPY for the full day (1 MWh = 1000 kWh).
    """
    if params is None:
        params = DEFAULT_PARAMS
    p = params
    T = len(price_vec)
    eta = np.sqrt(p["eta_rt"])
    soc_min = p["soc_min_frac"] * p["e_max"]
    soc_max = p["soc_max_frac"] * p["e_max"]
    soc_init = p["soc_init_frac"] * p["e_max"]
    useable = (p["soc_max_frac"] - p["soc_min_frac"]) * p["e_max"]
    cycle_cap_disch = useable * eta * p["max_cycles_per_day"]
    dt = p["dt"]

    charge = cp.Variable(T, nonneg=True)
    discharge = cp.Variable(T, nonneg=True)
    soc = cp.Variable(T + 1)

    # Revenue: (discharge - charge) [MW] * price [JPY/kWh] * dt [h] * 1000 [kWh/MWh]
    revenue = cp.sum(cp.multiply(discharge - charge, price_vec)) * dt * 1000.0

    cons = [
        charge <= p["p_max"],
        discharge <= p["p_max"],
        soc >= soc_min,
        soc <= soc_max,
        soc[0] == soc_init,
        cp.sum(discharge) <= cycle_cap_disch,
    ]
    for t in range(T):
        cons.append(soc[t + 1] == soc[t] + eta * charge[t] * dt - discharge[t] / eta * dt)

    prob = cp.Problem(cp.Maximize(revenue), cons)
    prob.solve(solver=cp.CLARABEL, verbose=False)

    return {
        "charge": charge.value,
        "discharge": discharge.value,
        "soc": soc.value,
        "revenue_forecast": float(prob.value) if prob.value is not None else 0.0,
        "status": prob.status,
    }


def dispatch_and_revalue(forecast_24h, realized_24h, params=None):
    """Solve LP using forecast prices, then revalue against realized prices.

    Returns:
        revenue_forecast : what the operator THINKS they'll earn
        revenue_realized : what they ACTUALLY earn (LP schedule × real prices)
    """
    if params is None:
        params = DEFAULT_PARAMS
    res = run_lp_day(forecast_24h, params)
    if res["charge"] is None:
        return {
            "revenue_forecast": 0.0,
            "revenue_realized": 0.0,
            "status": res["status"],
            "charge": None,
            "discharge": None,
        }
    realized_rev = float(
        np.sum((res["discharge"] - res["charge"]) * np.asarray(realized_24h)) * 1000.0
    )
    return {
        "revenue_forecast": res["revenue_forecast"],
        "revenue_realized": realized_rev,
        "status": res["status"],
        "charge": res["charge"],
        "discharge": res["discharge"],
    }

print("LP dispatch defined.")

## 2. Walk-Forward Splitter

Rolling 12-month train / 1-month test, advancing by 1 month per fold.

- Fold 1: train Jun 2022 → May 2023, test Jun 2023
- Fold 2: train Jul 2022 → Jun 2023, test Jul 2023
- ...
- Fold 25: train Jun 2024 → May 2025, test Jun 2025

Total: 25 folds, 761 test days.


In [ ]:
def walk_forward(df, ts_col="timestamp_jst", train_months=18, test_months=1,
                 step_months=1, min_train_months=None):
    """Yield (fold, train_df, test_df) tuples.

    If min_train_months is set, training expands (uses all data up to test_start).
    Otherwise rolling (fixed train_months window).
    """
    df = df.copy()
    df["_ts"] = pd.to_datetime(df[ts_col])
    start = df["_ts"].min().normalize()
    end = df["_ts"].max()

    fold = 0
    if min_train_months is not None:
        test_start = start + pd.DateOffset(months=min_train_months)
    else:
        test_start = start + pd.DateOffset(months=train_months)

    while True:
        test_end = test_start + pd.DateOffset(months=test_months)
        if test_end > end + pd.Timedelta(hours=1):
            break

        if min_train_months is not None:
            train_df = df[df["_ts"] < test_start]
        else:
            train_start = test_start - pd.DateOffset(months=train_months)
            train_df = df[(df["_ts"] >= train_start) & (df["_ts"] < test_start)]
        test_df = df[(df["_ts"] >= test_start) & (df["_ts"] < test_end)]

        yield fold, train_df.drop(columns="_ts"), test_df.drop(columns="_ts")

        fold += 1
        test_start = test_start + pd.DateOffset(months=step_months)


# Sanity check: count folds we'd get on the master
n_folds = sum(1 for _ in walk_forward(master, train_months=TRAIN_MONTHS,
                                       test_months=TEST_MONTHS, step_months=STEP_MONTHS))
print(f"Walk-forward yields {n_folds} folds on the master table.")

### Shared utilities

Two helpers used by every tier:

- `reshape_to_daily` / `long_to_daily_wide` — converts hourly rows into a (date × h00..h23) wide table for the LP.
- `evaluate_tier` — solves the LP on every test day and computes revenue per day.


In [ ]:
def reshape_to_daily(df, value_col, ts_col="timestamp_jst"):
    """Reshape hourly DataFrame into (date, h00..h23) wide format."""
    df = df.copy()
    df["_ts"] = pd.to_datetime(df[ts_col])
    df["date"] = df["_ts"].dt.date
    df["hour"] = df["_ts"].dt.hour
    out = df.pivot_table(index="date", columns="hour", values=value_col, aggfunc="first")
    out.columns = [f"h{h:02d}" for h in out.columns]
    return out.reset_index()


def long_to_daily_wide(long_df, value_col):
    """Same as reshape_to_daily, but takes a long-format DataFrame
    with columns [timestamp_jst, forecast, realized]."""
    df = long_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp_jst"])
    df["date"] = df["_ts"].dt.date
    df["hour"] = df["_ts"].dt.hour
    out = df.pivot_table(index="date", columns="hour", values=value_col, aggfunc="first")
    out.columns = [f"h{h:02d}" for h in out.columns]
    return out.reset_index()


def evaluate_tier(forecast_daily, realized_daily, label, verbose=True):
    """Solve LP per day with forecast, revalue against realized prices.

    Returns a DataFrame with columns: date, revenue_forecast, revenue_realized, status.
    """
    merged = forecast_daily.merge(realized_daily, on="date", suffixes=("_f", "_r"))
    rows = []
    t0 = time.time()
    for _, r in merged.iterrows():
        f24 = np.array([r[f"h{h:02d}_f"] for h in range(24)])
        a24 = np.array([r[f"h{h:02d}_r"] for h in range(24)])
        if np.isnan(f24).any() or np.isnan(a24).any():
            continue
        d = dispatch_and_revalue(f24, a24)
        rows.append({
            "date": r["date"],
            "revenue_forecast": d["revenue_forecast"],
            "revenue_realized": d["revenue_realized"],
            "status": d["status"],
        })
    elapsed = time.time() - t0
    if verbose:
        per = elapsed / max(len(rows), 1) * 1000
        print(f"  [{label}] solved {len(rows)} days in {elapsed:.1f}s ({per:.0f} ms/day)")
    out = pd.DataFrame(rows)
    out["date"] = pd.to_datetime(out["date"]).dt.date
    return out


def summarize_tier(rev_df, pf_df, label):
    """Print headline numbers and return a comparison DataFrame."""
    cmp = rev_df.merge(
        pf_df[["date", "revenue_jpy"]].rename(columns={"revenue_jpy": "revenue_pf"}),
        on="date", how="inner"
    )
    cmp["rer"] = cmp["revenue_realized"] / cmp["revenue_pf"].replace(0, np.nan)
    cmp["year"] = pd.to_datetime(cmp["date"]).dt.year

    total_r = cmp["revenue_realized"].sum()
    total_pf = cmp["revenue_pf"].sum()
    capture = total_r / total_pf * 100
    neg = (cmp["revenue_realized"] < 0).sum()
    var5 = cmp["revenue_realized"].quantile(0.05)
    cvar5 = cmp.loc[cmp["revenue_realized"] <= var5, "revenue_realized"].mean()
    mean = cmp["revenue_realized"].mean()

    print(f"\n=== {label} headline ===")
    print(f"  Total realized   : {total_r:>16,.0f} JPY")
    print(f"  Total perfect    : {total_pf:>16,.0f} JPY")
    print(f"  Capture %        : {capture:>15.1f}%")
    print(f"  Mean daily rev   : {mean:>16,.0f} JPY")
    print(f"  Negative-rev days: {neg:>16d} ({neg/len(cmp)*100:.1f}%)")
    print(f"  5% VaR (RaR)     : {var5:>16,.0f} JPY")
    print(f"  5% CVaR          : {cvar5:>16,.0f} JPY")

    by_year = cmp.groupby("year").agg(
        days=("date", "count"),
        rev=("revenue_realized", "sum"),
        rev_pf=("revenue_pf", "sum"),
    )
    by_year["capture_%"] = (by_year["rev"] / by_year["rev_pf"] * 100).round(1)
    print(f"\n  By year:")
    print(by_year.to_string())
    return cmp

print("Shared utilities defined.")

## 3. T0 — Naive lag-168h Forecast

**The "do nothing" baseline.** Forecast for hour *h* of date *d* is simply the realized price at hour *h* of date *d* − 7 days.

- 1 feature
- 0 lines of training code (just a column lookup)
- Drops first week (no lag available)


In [ ]:
def t0_naive_forecast(test_df, price_col="price_tokyo", lag_col="price_lag_168h"):
    """For every hour in test_df, forecast = price_lag_168h."""
    test = test_df.copy()
    if test[lag_col].isna().any():
        test[lag_col] = test[lag_col].fillna(
            test.groupby("hour")[price_col].transform("mean"))
    return reshape_to_daily(test, lag_col)


def realized_to_daily(test_df, price_col="price_tokyo"):
    return reshape_to_daily(test_df, price_col)

In [ ]:
# T0 doesn't strictly need walk-forward (no training), but we apply it on the
# whole post-first-week window to mirror downstream tiers' coverage.
print("=== T0: Naive lag-168h ===")
m_t0 = master.dropna(subset=["price_lag_168h"]).copy()
fcst_t0 = t0_naive_forecast(m_t0)
real_t0 = realized_to_daily(m_t0)
print(f"  Forecast covers {len(fcst_t0)} days")

rev_t0 = evaluate_tier(fcst_t0, real_t0, "T0")
rev_t0.to_csv(f"{RESULTS_DIR}/E1_t0_revenue.csv", index=False)
cmp_t0 = summarize_tier(rev_t0, pf, "T0")
cmp_t0.to_csv(f"{RESULTS_DIR}/E1_t0_comparison.csv", index=False)

## 4. T1 — Per-hour OLS with Calendar (ARX-style)

For each hour-of-day *h* ∈ {0, …, 23}, fit a separate OLS:

$$ p_{h, t} = \beta_0 + \beta_1 p_{h, t-24} + \beta_2 p_{h, t-168} + \beta_3 p_{h, t-1} + \sum_{w=1}^{6} \delta_w \mathbb{1}(dow=w) + \beta_4 \mathbb{1}(\text{off-day}) + \epsilon $$

- 24 separate models (per hour-of-day)
- 10 features each
- Refit every fold (≈ ms per fit)
- This is the standard univariate electricity-price benchmark (Weron 2014 ARX).


In [ ]:
T1_FEATS = [
    "price_lag_24h", "price_lag_168h", "price_lag_1h", "is_offday",
    "dow_1", "dow_2", "dow_3", "dow_4", "dow_5", "dow_6",
]


def _fit_one_hour_ols(train_h, feats):
    X = train_h[feats].values
    y = train_h["price_tokyo"].values
    mask = ~np.isnan(X).any(axis=1) & ~np.isnan(y)
    if mask.sum() < 30:
        return None
    return LinearRegression().fit(X[mask], y[mask])


def make_t1_forecasts(master_df, verbose=True):
    rows, fold_log = [], []
    df = master_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp_jst"])
    for d in range(1, 7):
        df[f"dow_{d}"] = (df["dow"] == d).astype(int)

    for fold, train_df, test_df in walk_forward(
        df, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS, step_months=STEP_MONTHS
    ):
        train_df = train_df.copy(); test_df = test_df.copy()
        train_df["_ts"] = pd.to_datetime(train_df["timestamp_jst"])
        test_df["_ts"] = pd.to_datetime(test_df["timestamp_jst"])

        t0 = time.time()
        models = {h: _fit_one_hour_ols(train_df[train_df["_ts"].dt.hour == h], T1_FEATS)
                  for h in range(24)}
        fit_t = time.time() - t0

        t1 = time.time()
        for _, r in test_df.iterrows():
            h = int(r["hour"])
            mdl = models[h]
            if mdl is None: continue
            x = np.array([r[f] for f in T1_FEATS], dtype=float)
            if np.isnan(x).any(): continue
            pred = float(np.clip(mdl.predict(x.reshape(1, -1))[0], PRICE_FLOOR, PRICE_CEIL))
            rows.append({"timestamp_jst": r["timestamp_jst"],
                         "forecast": pred, "realized": float(r["price_tokyo"])})
        fcst_t = time.time() - t1

        if verbose and fold % 5 == 0:
            print(f"  Fold {fold:>2d}  test {test_df['_ts'].min().date()}  "
                  f"fit {fit_t*1000:.0f}ms  fcst {fcst_t*1000:.0f}ms")
        fold_log.append({"fold": fold,
                         "train_start": train_df["_ts"].min().date(),
                         "test_start": test_df["_ts"].min().date(),
                         "fit_seconds": fit_t, "forecast_seconds": fcst_t,
                         "n_test_days": test_df["_ts"].dt.date.nunique()})
    return pd.DataFrame(rows), pd.DataFrame(fold_log)

In [ ]:
print("=== T1: Per-hour OLS with calendar ===")
t1_long, t1_fold_log = make_t1_forecasts(master)
t1_long.to_csv(f"{RESULTS_DIR}/E2_t1_forecasts_long.csv", index=False)
t1_fold_log.to_csv(f"{RESULTS_DIR}/E2_t1_fold_log.csv", index=False)
print(f"  {len(t1_long):,} forecast rows from {len(t1_fold_log)} folds")

# LP + summary
fcst_t1 = long_to_daily_wide(t1_long, "forecast")
real_t1 = long_to_daily_wide(t1_long, "realized")
rev_t1 = evaluate_tier(fcst_t1, real_t1, "T1")
rev_t1.to_csv(f"{RESULTS_DIR}/E2_t1_revenue.csv", index=False)
cmp_t1 = summarize_tier(rev_t1, pf, "T1")
cmp_t1.to_csv(f"{RESULTS_DIR}/E2_t1_comparison.csv", index=False)

## 5. T2 — OLS + Exogenous Fundamentals

Same per-hour OLS structure as T1, but feature set grows by 4 contemporaneous fundamentals:

- `load` — TEPCO area load
- `pv_actual` — PV generation
- `temp_c` — Tokyo temperature
- `lng_japan_usd_mmbtu` — **daily JKM LNG price**

Total: 14 features. The purpose of T2 is to isolate the marginal value of adding fundamental drivers, holding model class fixed.

**Note** — LNG here is daily-frequency (`lng_japan_usd_mmbtu`), not the monthly `lng_japan_monthly_old`.


In [ ]:
T2_FEATS = [
    # T1 carry-over
    "price_lag_24h", "price_lag_168h", "price_lag_1h", "is_offday",
    "dow_1", "dow_2", "dow_3", "dow_4", "dow_5", "dow_6",
    # T2 NEW: contemporaneous fundamentals
    "load", "pv_actual", "temp_c", "lng_japan_usd_mmbtu",
]


def make_t2_forecasts(master_df, verbose=True):
    rows, fold_log = [], []
    df = master_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp_jst"])
    for d in range(1, 7):
        df[f"dow_{d}"] = (df["dow"] == d).astype(int)

    for fold, train_df, test_df in walk_forward(
        df, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS, step_months=STEP_MONTHS,
    ):
        train_df = train_df.copy(); test_df = test_df.copy()
        train_df["_ts"] = pd.to_datetime(train_df["timestamp_jst"])
        test_df["_ts"] = pd.to_datetime(test_df["timestamp_jst"])

        t0 = time.time()
        models = {h: _fit_one_hour_ols(train_df[train_df["_ts"].dt.hour == h], T2_FEATS)
                  for h in range(24)}
        fit_t = time.time() - t0

        t1 = time.time()
        for _, r in test_df.iterrows():
            h = int(r["hour"])
            mdl = models[h]
            if mdl is None: continue
            x = np.array([r[f] for f in T2_FEATS], dtype=float)
            if np.isnan(x).any(): continue
            pred = float(np.clip(mdl.predict(x.reshape(1, -1))[0], PRICE_FLOOR, PRICE_CEIL))
            rows.append({"timestamp_jst": r["timestamp_jst"],
                         "forecast": pred, "realized": float(r["price_tokyo"])})
        fcst_t = time.time() - t1

        if verbose and fold % 5 == 0:
            print(f"  Fold {fold:>2d}  test {test_df['_ts'].min().date()}  "
                  f"fit {fit_t*1000:.0f}ms  fcst {fcst_t*1000:.0f}ms")
        fold_log.append({"fold": fold,
                         "train_start": train_df["_ts"].min().date(),
                         "test_start": test_df["_ts"].min().date(),
                         "fit_seconds": fit_t, "forecast_seconds": fcst_t,
                         "n_test_days": test_df["_ts"].dt.date.nunique()})
    return pd.DataFrame(rows), pd.DataFrame(fold_log)

In [ ]:
print("=== T2: OLS + exogenous (load, PV, temp, LNG) ===")
t2_long, t2_fold_log = make_t2_forecasts(master)
t2_long.to_csv(f"{RESULTS_DIR}/E3_t2_forecasts_long.csv", index=False)
t2_fold_log.to_csv(f"{RESULTS_DIR}/E3_t2_fold_log.csv", index=False)

fcst_t2 = long_to_daily_wide(t2_long, "forecast")
real_t2 = long_to_daily_wide(t2_long, "realized")
rev_t2 = evaluate_tier(fcst_t2, real_t2, "T2")
rev_t2.to_csv(f"{RESULTS_DIR}/E3_t2_revenue.csv", index=False)
cmp_t2 = summarize_tier(rev_t2, pf, "T2")
cmp_t2.to_csv(f"{RESULTS_DIR}/E3_t2_comparison.csv", index=False)

## 6. T3 — LightGBM (Gradient Boosted Trees)

Same data scope as T2, but a non-linear model that can capture interactions like
*high temp ∧ low PV ∧ high LNG → spike*.

**Architecture choice** — single global model (not per-hour). LightGBM handles
hour-of-day effects through features (`hour`, `dow`, `month`), which is closer
to practitioner setups and reduces fit overhead.

**Features (18 total):**
- Lag: `price_lag_1h`, `price_lag_24h`, `price_lag_168h`, rolling 24h mean/std, load/PV lag
- Calendar: `hour`, `dow`, `month`, `is_weekend`, `is_holiday`, `is_offday`
- Fundamentals: `load`, `pv_actual`, `temp_c`, `sunshine_h`, `lng_japan_usd_mmbtu`

**Hyperparams** — fixed across folds for effort-comparable evaluation:
- Objective: `regression_l1` (MAE, robust to spikes)
- 500 trees max, early stopping on last 14-day validation slice


In [ ]:
T3_FEATS = [
    # lag
    "price_lag_1h", "price_lag_24h", "price_lag_168h",
    "price_roll_24h_mean", "price_roll_24h_std",
    "load_lag_24h", "pv_actual_lag_24h",
    # calendar
    "hour", "dow", "month", "is_weekend", "is_holiday", "is_offday",
    # contemporaneous fundamentals
    "load", "pv_actual", "temp_c", "sunshine_h", "lng_japan_usd_mmbtu",
]

LGB_REG_PARAMS = dict(
    objective="regression_l1",
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=50,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    verbose=-1,
    n_estimators=500,
)


def make_t3_forecasts(master_df, verbose=True):
    rows, fold_log, feat_imp = [], [], []
    df = master_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp_jst"])

    for fold, train_df, test_df in walk_forward(
        df, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS, step_months=STEP_MONTHS,
    ):
        train_df = train_df.copy(); test_df = test_df.copy()
        train_df["_ts"] = pd.to_datetime(train_df["timestamp_jst"])
        test_df["_ts"] = pd.to_datetime(test_df["timestamp_jst"])

        # Hold out last 14 days for early stopping
        val_start = train_df["_ts"].max() - pd.Timedelta(days=14)
        tr = train_df[train_df["_ts"] < val_start]
        va = train_df[train_df["_ts"] >= val_start]

        Xtr = tr[T3_FEATS].values; ytr = tr["price_tokyo"].values
        Xva = va[T3_FEATS].values; yva = va["price_tokyo"].values
        m_tr = ~np.isnan(Xtr).any(axis=1) & ~np.isnan(ytr)
        m_va = ~np.isnan(Xva).any(axis=1) & ~np.isnan(yva)

        t0 = time.time()
        model = lgb.LGBMRegressor(**LGB_REG_PARAMS)
        model.fit(Xtr[m_tr], ytr[m_tr],
                  eval_set=[(Xva[m_va], yva[m_va])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        fit_t = time.time() - t0

        t1 = time.time()
        Xte = test_df[T3_FEATS].values
        m_te = ~np.isnan(Xte).any(axis=1)
        yhat = np.full(len(test_df), np.nan)
        yhat[m_te] = np.clip(model.predict(Xte[m_te]), PRICE_FLOOR, PRICE_CEIL)
        for i, (_, r) in enumerate(test_df.iterrows()):
            if not np.isnan(yhat[i]):
                rows.append({"timestamp_jst": r["timestamp_jst"],
                             "forecast": float(yhat[i]),
                             "realized": float(r["price_tokyo"])})
        fcst_t = time.time() - t1

        for f, imp in zip(T3_FEATS, model.feature_importances_):
            feat_imp.append({"fold": fold, "feature": f, "importance": int(imp)})

        if verbose and fold % 5 == 0:
            print(f"  Fold {fold:>2d}  test {test_df['_ts'].min().date()}  "
                  f"fit {fit_t:.1f}s ({model.best_iteration_} iters)  fcst {fcst_t*1000:.0f}ms")
        fold_log.append({"fold": fold,
                         "train_start": train_df["_ts"].min().date(),
                         "test_start": test_df["_ts"].min().date(),
                         "fit_seconds": fit_t, "forecast_seconds": fcst_t,
                         "best_iter": int(model.best_iteration_),
                         "n_test_days": test_df["_ts"].dt.date.nunique()})

    return pd.DataFrame(rows), pd.DataFrame(fold_log), pd.DataFrame(feat_imp)

In [ ]:
print("=== T3: LightGBM ===")
t3_long, t3_fold_log, t3_feat_imp = make_t3_forecasts(master)
t3_long.to_csv(f"{RESULTS_DIR}/E4_t3_forecasts_long.csv", index=False)
t3_fold_log.to_csv(f"{RESULTS_DIR}/E4_t3_fold_log.csv", index=False)
t3_feat_imp.to_csv(f"{RESULTS_DIR}/E4_t3_feature_importance.csv", index=False)

fcst_t3 = long_to_daily_wide(t3_long, "forecast")
real_t3 = long_to_daily_wide(t3_long, "realized")
rev_t3 = evaluate_tier(fcst_t3, real_t3, "T3")
rev_t3.to_csv(f"{RESULTS_DIR}/E4_t3_revenue.csv", index=False)
cmp_t3 = summarize_tier(rev_t3, pf, "T3")
cmp_t3.to_csv(f"{RESULTS_DIR}/E4_t3_comparison.csv", index=False)

# Feature importance summary
print("\n  Top 5 features (avg across folds):")
print(t3_feat_imp.groupby("feature")["importance"].mean().sort_values(ascending=False).head().to_string())

## 7. T4 — T3 + Top-K Spike Booster

**Hypothesis** — T3 systematically under-predicts the daily high-hours
(GBM's right-tail smoothing bias). A learnable remedy:

1. Train a separate LightGBM **classifier** that flags "this hour will rank in
   the top-K of its day".
2. For positive flags, multiply the T3 forecast by α.

Both **K ∈ {2, 3, 4}** and **α ∈ {1.05, 1.10, 1.15, 1.20, 1.25, 1.30}** are
tuned per fold on a 14-day validation slice — so T4's hyperparams are NOT
fixed at α=1.10. The fold log records what was selected per fold.

Despite this per-fold tuning, T4 still underperforms T3 in our results.


In [ ]:
TOP_K_GRID = [2, 3, 4]
ALPHA_GRID = [1.05, 1.10, 1.15, 1.20, 1.25, 1.30]

LGB_CLF_PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    num_leaves=32,
    min_data_in_leaf=80,
    feature_fraction=0.9,
    verbose=-1,
    n_estimators=300,
)


def _label_top_k(df, k):
    """Per-day: y=1 if hour is in top-K of that day's prices."""
    d = df.copy()
    d["_ts"] = pd.to_datetime(d["timestamp_jst"])
    d["date"] = d["_ts"].dt.date
    d["rank"] = d.groupby("date")["price_tokyo"].rank(ascending=False, method="first")
    return (d["rank"] <= k).astype(int).values


def _train_t3_reg(tr, va):
    Xtr = tr[T3_FEATS].values; ytr = tr["price_tokyo"].values
    Xva = va[T3_FEATS].values; yva = va["price_tokyo"].values
    mtr = ~np.isnan(Xtr).any(axis=1) & ~np.isnan(ytr)
    mva = ~np.isnan(Xva).any(axis=1) & ~np.isnan(yva)
    m = lgb.LGBMRegressor(**LGB_REG_PARAMS)
    m.fit(Xtr[mtr], ytr[mtr], eval_set=[(Xva[mva], yva[mva])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return m


def _train_top_k_clf(tr, k):
    y = _label_top_k(tr, k)
    X = tr[T3_FEATS].values
    m = ~np.isnan(X).any(axis=1)
    pos_w = (1 - y[m].mean()) / y[m].mean() if y[m].mean() > 0 else 1.0
    clf = lgb.LGBMClassifier(scale_pos_weight=pos_w, **LGB_CLF_PARAMS)
    clf.fit(X[m], y[m])
    return clf


def _apply_boost(forecast, proba, alpha, threshold=0.5):
    out = forecast.copy()
    mask = proba >= threshold
    out[mask] = np.clip(out[mask] * alpha, PRICE_FLOOR, PRICE_CEIL)
    return out


def make_t4_forecasts(master_df, verbose=True):
    rows, fold_log = [], []
    df = master_df.copy()
    df["_ts"] = pd.to_datetime(df["timestamp_jst"])

    for fold, train_df, test_df in walk_forward(
        df, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS, step_months=STEP_MONTHS,
    ):
        train_df = train_df.copy(); test_df = test_df.copy()
        train_df["_ts"] = pd.to_datetime(train_df["timestamp_jst"])
        test_df["_ts"] = pd.to_datetime(test_df["timestamp_jst"])

        val_start = train_df["_ts"].max() - pd.Timedelta(days=14)
        tr = train_df[train_df["_ts"] < val_start]
        va = train_df[train_df["_ts"] >= val_start]

        t0 = time.time()
        # 1. Regressor (same as T3)
        reg = _train_t3_reg(tr, va)
        Xte = test_df[T3_FEATS].values
        m_te = ~np.isnan(Xte).any(axis=1)
        yhat_t3 = np.full(len(test_df), np.nan)
        yhat_t3[m_te] = np.clip(reg.predict(Xte[m_te]), PRICE_FLOOR, PRICE_CEIL)

        # 2. Validation forecast (for tuning K, alpha)
        Xva = va[T3_FEATS].values
        m_va = ~np.isnan(Xva).any(axis=1) & ~np.isnan(va["price_tokyo"].values)
        yhat_va = np.full(len(va), np.nan)
        yhat_va[m_va] = np.clip(reg.predict(Xva[m_va]), PRICE_FLOOR, PRICE_CEIL)

        # 3. Grid search K & alpha — score = MAE on top-K hours of each day
        best_score, best_k, best_alpha = np.inf, TOP_K_GRID[0], 1.0
        for k in TOP_K_GRID:
            clf = _train_top_k_clf(tr, k)
            proba_va = np.full(len(va), np.nan)
            proba_va[m_va] = clf.predict_proba(Xva[m_va])[:, 1]
            for alpha in ALPHA_GRID:
                ymod = _apply_boost(yhat_va.copy(), proba_va, alpha)
                df_va = va.copy()
                df_va["yhat"] = ymod
                df_va["yreal"] = va["price_tokyo"].values
                df_va["date"] = df_va["_ts"].dt.date
                df_va["rank_real"] = df_va.groupby("date")["yreal"].rank(ascending=False, method="first")
                top = df_va[df_va["rank_real"] <= k]
                score = float(np.nanmean(np.abs(top["yhat"] - top["yreal"])))
                if score < best_score:
                    best_score, best_k, best_alpha = score, k, alpha

        # 4. Train final classifier with best K, apply to test
        clf_final = _train_top_k_clf(tr, best_k)
        proba_te = np.full(len(test_df), np.nan)
        proba_te[m_te] = clf_final.predict_proba(Xte[m_te])[:, 1]
        yhat_t4 = _apply_boost(yhat_t3, proba_te, best_alpha)

        fit_t = time.time() - t0

        for i, (_, r) in enumerate(test_df.iterrows()):
            if np.isnan(yhat_t4[i]): continue
            rows.append({"timestamp_jst": r["timestamp_jst"],
                         "forecast": float(yhat_t4[i]),
                         "realized": float(r["price_tokyo"])})

        if verbose and fold % 5 == 0:
            n_days = test_df["_ts"].dt.date.nunique()
            n_boost = int((proba_te[m_te] >= 0.5).sum())
            print(f"  Fold {fold:>2d}  test {test_df['_ts'].min().date()}  "
                  f"K={best_k} alpha={best_alpha:.2f}  boost {n_boost}/{m_te.sum()} hrs  "
                  f"fit {fit_t:.1f}s ({n_days}d)")
        fold_log.append({"fold": fold,
                         "train_start": train_df["_ts"].min().date(),
                         "test_start": test_df["_ts"].min().date(),
                         "best_k": best_k, "best_alpha": best_alpha,
                         "fit_seconds": fit_t,
                         "n_test_days": test_df["_ts"].dt.date.nunique()})
    return pd.DataFrame(rows), pd.DataFrame(fold_log)

In [ ]:
print("=== T4: T3 + spike-booster (per-fold tuned K, α) ===")
t4_long, t4_fold_log = make_t4_forecasts(master)
t4_long.to_csv(f"{RESULTS_DIR}/E5_t4_forecasts_long.csv", index=False)
t4_fold_log.to_csv(f"{RESULTS_DIR}/E5_t4_fold_log.csv", index=False)

fcst_t4 = long_to_daily_wide(t4_long, "forecast")
real_t4 = long_to_daily_wide(t4_long, "realized")
rev_t4 = evaluate_tier(fcst_t4, real_t4, "T4")
rev_t4.to_csv(f"{RESULTS_DIR}/E5_t4_revenue.csv", index=False)
cmp_t4 = summarize_tier(rev_t4, pf, "T4")
cmp_t4.to_csv(f"{RESULTS_DIR}/E5_t4_comparison.csv", index=False)

# Inspect tuned hyperparams per fold
print("\n  Tuned hyperparams per fold:")
print(t4_fold_log[["fold", "test_start", "best_k", "best_alpha"]].to_string(index=False))

## 8. Aggregate Comparison Across Tiers

Side-by-side summary: capture rate, mean revenue, 5% RaR, negative-revenue days.
This is the table that drives the headline result in the slides and report.


In [ ]:
tier_summaries = []
for label, cmp in [("T0", cmp_t0), ("T1", cmp_t1), ("T2", cmp_t2),
                   ("T3", cmp_t3), ("T4", cmp_t4)]:
    total_r = cmp["revenue_realized"].sum()
    total_pf = cmp["revenue_pf"].sum()
    var5 = cmp["revenue_realized"].quantile(0.05)
    cvar5 = cmp.loc[cmp["revenue_realized"] <= var5, "revenue_realized"].mean()
    tier_summaries.append({
        "tier": label,
        "n_days": len(cmp),
        "capture_%": round(total_r / total_pf * 100, 1),
        "mean_rev_JPY": round(cmp["revenue_realized"].mean()),
        "5pct_RaR_JPY": round(var5),
        "5pct_CVaR_JPY": round(cvar5),
        "neg_days": (cmp["revenue_realized"] < 0).sum(),
    })

summary = pd.DataFrame(tier_summaries)
summary.to_csv(f"{RESULTS_DIR}/streamB_summary.csv", index=False)
print(summary.to_string(index=False))

In [ ]:
# Marginal ROI per upgrade (T0→T1, T1→T2, T2→T3, T3→T4)
roi_rows = []
for prev, curr in [("T0", "T1"), ("T1", "T2"), ("T2", "T3"), ("T3", "T4")]:
    p = summary[summary["tier"] == prev].iloc[0]
    c = summary[summary["tier"] == curr].iloc[0]
    roi_rows.append({
        "upgrade": f"{prev} → {curr}",
        "delta_capture_pp": round(c["capture_%"] - p["capture_%"], 2),
        "delta_mean_rev": int(c["mean_rev_JPY"] - p["mean_rev_JPY"]),
        "delta_RaR": int(c["5pct_RaR_JPY"] - p["5pct_RaR_JPY"]),
        "delta_neg_days": int(c["neg_days"] - p["neg_days"]),
    })
roi = pd.DataFrame(roi_rows)
roi.to_csv(f"{RESULTS_DIR}/streamB_roi.csv", index=False)
print("\nMarginal ROI by upgrade:")
print(roi.to_string(index=False))

## 9. Visualizations

Adapted from `make_figs.py`. Four figures useful for slides:

1. **RER histogram** — daily revenue efficiency ratio distribution for each tier
2. **Cumulative revenue** — T0, T1, T3 vs PF
3. **Capture % by year** — robustness across regime
4. **Negative-revenue days** — share of money-losing days by tier


In [ ]:
# Fig 1: RER histogram for T0 (the most dramatic)
fig, ax = plt.subplots(figsize=(8, 4.5))
rer_clip = cmp_t0["rer"].clip(-2, 1.2)
ax.hist(rer_clip, bins=60, color="#3F7CAC", edgecolor="white", alpha=0.85)
ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="Zero (no profit)")
ax.axvline(1, color="green", linestyle="--", linewidth=1.2, label="Perfect foresight")
ax.axvline(cmp_t0["rer"].median(), color="black", linestyle="-", linewidth=1.5,
           label=f"Median = {cmp_t0['rer'].median():.2f}")
ax.set_xlabel("Daily RER (realized / perfect-foresight)")
ax.set_ylabel("Number of days")
neg = (cmp_t0["revenue_realized"] < 0).sum()
ax.set_title(f"T0 Naive: distribution of daily Revenue Efficiency Ratio\n"
             f"({neg} days lose money; median RER {cmp_t0['rer'].median():.2f})", fontsize=11)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_T0_rer_histogram.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fig 2: Cumulative revenue — T0, T1, T3 vs PF
fig, ax = plt.subplots(figsize=(9.5, 4.5))
for cmp_df, label, color in [
    (cmp_t0, "T0 Naive", "#C62828"),
    (cmp_t1, "T1 OLS",   "#3F7CAC"),
    (cmp_t3, "T3 LGBM",  "#2E7D32"),
]:
    df = cmp_df.sort_values("date").copy()
    df["date"] = pd.to_datetime(df["date"])
    df["cum"] = df["revenue_realized"].cumsum() / 1e6
    ax.plot(df["date"], df["cum"], label=label, color=color, linewidth=2)

df_pf = cmp_t0.sort_values("date").copy()
df_pf["date"] = pd.to_datetime(df_pf["date"])
df_pf["cum_pf"] = df_pf["revenue_pf"].cumsum() / 1e6
ax.plot(df_pf["date"], df_pf["cum_pf"], label="Perfect Foresight",
        color="black", linewidth=1.5, linestyle="--", alpha=0.6)

ax.set_ylabel("Cumulative revenue (M JPY)")
ax.set_title("Cumulative arbitrage revenue across forecast tiers")
ax.legend(frameon=False, loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_cumulative_revenue.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fig 3: Capture % by year
fig, ax = plt.subplots(figsize=(9, 4.5))
width = 0.18
years = sorted(cmp_t0["year"].unique())
x = np.arange(len(years))

for i, (cmp_df, label, color) in enumerate([
    (cmp_t0, "T0", "#C62828"),
    (cmp_t1, "T1", "#3F7CAC"),
    (cmp_t2, "T2", "#7E57C2"),
    (cmp_t3, "T3", "#2E7D32"),
    (cmp_t4, "T4", "#F57C00"),
]):
    by_year = cmp_df.groupby("year").agg(
        rev=("revenue_realized", "sum"), pf=("revenue_pf", "sum"))
    by_year["capture"] = by_year["rev"] / by_year["pf"] * 100
    vals = [by_year.loc[y, "capture"] if y in by_year.index else 0 for y in years]
    ax.bar(x + (i - 2) * width, vals, width, label=label, color=color, edgecolor="white")

ax.axhline(100, color="black", linestyle="--", linewidth=1, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_ylabel("% of perfect-foresight revenue captured")
ax.set_title("Capture rate by year × tier")
ax.set_ylim(0, 110)
ax.legend(frameon=False, ncol=5, loc="lower center")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_capture_by_year.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fig 4: Negative-revenue days
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ["T0", "T1", "T2", "T3", "T4"]
vals = []
for label, cmp_df in [("T0", cmp_t0), ("T1", cmp_t1), ("T2", cmp_t2),
                      ("T3", cmp_t3), ("T4", cmp_t4)]:
    pct = (cmp_df["revenue_realized"] < 0).mean() * 100
    n = (cmp_df["revenue_realized"] < 0).sum()
    vals.append((pct, n))

bars = ax.bar(labels, [v[0] for v in vals],
              color=["#C62828", "#3F7CAC", "#7E57C2", "#2E7D32", "#F57C00"],
              edgecolor="white", width=0.55)
for bar, (pct, n) in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, pct + 0.4,
            f"{pct:.1f}%\n({n} days)",
            ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Share of days with negative revenue (%)")
ax.set_title("Negative-revenue days by forecast tier")
ax.set_ylim(0, max(v[0] for v in vals) * 1.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_negative_days_by_tier.png", bbox_inches="tight")
plt.show()